In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import time

# Load the preprocessed data from Phase 2
X_train = pd.read_csv('../data/X_train_scaled.csv')
X_test = pd.read_csv('../data/X_test_scaled.csv')
y_train = pd.read_csv('../data/y_train.csv')['target']
y_test = pd.read_csv('../data/y_test.csv')['target']

# Load the target encoder to see what the numbers mean
target_encoder = joblib.load('../models/target_encoder.joblib')
print(f"Classes: {dict(zip(range(len(target_encoder.classes_)), target_encoder.classes_))}")
print(f"\nTraining set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"\nTarget distribution (train):")
for i, name in enumerate(target_encoder.classes_):
    count = (y_train == i).sum()
    print(f"  {name}: {count}")

Classes: {0: np.str_('DoS'), 1: np.str_('Normal'), 2: np.str_('Probe'), 3: np.str_('R2L'), 4: np.str_('U2R')}

Training set: 125973 samples, 41 features
Test set:     22544 samples, 41 features

Target distribution (train):
  DoS: 45927
  Normal: 67343
  Probe: 11656
  R2L: 995
  U2R: 52


In [3]:
# BASELINE MODEL — default hyperparameters
# This is our "before" picture. We'll improve it next.

print("Training baseline Random Forest...")
start_time = time.time()

baseline_rf = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees in the forest
    random_state=42,     # makes results reproducible
    n_jobs=-1            # use all CPU cores for speed
)

baseline_rf.fit(X_train, y_train)
train_time = time.time() - start_time

print(f"Training complete in {train_time:.1f} seconds")
print(f"Number of trees: {baseline_rf.n_estimators}")
print(f"Number of features: {baseline_rf.n_features_in_}")

Training baseline Random Forest...
Training complete in 1.2 seconds
Number of trees: 100
Number of features: 41


In [4]:
# Predict on both train and test sets
y_train_pred = baseline_rf.predict(X_train)
y_test_pred = baseline_rf.predict(X_test)

# Overall accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training accuracy: {train_acc*100:.2f}%")
print(f"Test accuracy:     {test_acc*100:.2f}%")
print(f"\nGap: {(train_acc - test_acc)*100:.2f}%")
print("(A large gap means overfitting — the model memorized training data")
print(" instead of learning general patterns)")

Training accuracy: 99.99%
Test accuracy:     75.12%

Gap: 24.87%
(A large gap means overfitting — the model memorized training data
 instead of learning general patterns)


In [5]:
# Detailed classification report
# This is what you'd show in an interview — not just "accuracy"

print("=" * 60)
print("CLASSIFICATION REPORT — TEST SET")
print("=" * 60)
target_names = target_encoder.classes_
print(classification_report(y_test, y_test_pred, target_names=target_names))

CLASSIFICATION REPORT — TEST SET
              precision    recall  f1-score   support

         DoS       0.96      0.77      0.85      7458
      Normal       0.65      0.97      0.78      9711
       Probe       0.87      0.67      0.76      2421
         R2L       0.98      0.04      0.08      2887
         U2R       0.50      0.03      0.06        67

    accuracy                           0.75     22544
   macro avg       0.79      0.50      0.51     22544
weighted avg       0.82      0.75      0.71     22544



In [6]:
# TUNED MODEL — fighting overfitting
#
# Key changes:
# - max_depth: limits how deep each tree can grow (prevents memorization)
# - min_samples_split: requires more samples before a tree can split
# - min_samples_leaf: requires more samples in each leaf node
# - class_weight: tells the model to pay MORE attention to rare classes
#   This is the big one — it multiplies the penalty for misclassifying
#   rare classes like U2R and R2L

print("Training tuned Random Forest...")
start_time = time.time()

tuned_rf = RandomForestClassifier(
    n_estimators=200,          # more trees for better voting
    max_depth=30,              # limit tree depth to prevent memorization
    min_samples_split=5,       # need at least 5 samples to make a split
    min_samples_leaf=2,        # each leaf must have at least 2 samples
    class_weight='balanced',   # THE KEY: automatically adjust weights
    random_state=42,
    n_jobs=-1
)

tuned_rf.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Training complete in {train_time:.1f} seconds")

Training tuned Random Forest...
Training complete in 2.1 seconds


In [7]:
# Evaluate tuned model
y_train_pred_tuned = tuned_rf.predict(X_train)
y_test_pred_tuned = tuned_rf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred_tuned)
test_acc = accuracy_score(y_test, y_test_pred_tuned)

print(f"Training accuracy: {train_acc*100:.2f}%")
print(f"Test accuracy:     {test_acc*100:.2f}%")
print(f"Gap: {(train_acc - test_acc)*100:.2f}%")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT — TUNED MODEL (TEST SET)")
print("=" * 60)
print(classification_report(y_test, y_test_pred_tuned, target_names=target_encoder.classes_))

Training accuracy: 99.97%
Test accuracy:     73.92%
Gap: 26.05%

CLASSIFICATION REPORT — TUNED MODEL (TEST SET)
              precision    recall  f1-score   support

         DoS       0.96      0.76      0.85      7458
      Normal       0.64      0.97      0.77      9711
       Probe       0.84      0.62      0.72      2421
         R2L       0.78      0.00      0.01      2887
         U2R       0.50      0.07      0.13        67

    accuracy                           0.74     22544
   macro avg       0.74      0.49      0.50     22544
weighted avg       0.78      0.74      0.69     22544



In [8]:
# Side-by-side comparison
from sklearn.metrics import f1_score

baseline_f1 = f1_score(y_test, y_test_pred, average=None)
tuned_f1 = f1_score(y_test, y_test_pred_tuned, average=None)

comparison = pd.DataFrame({
    'Class': target_encoder.classes_,
    'Baseline F1': baseline_f1.round(3),
    'Tuned F1': tuned_f1.round(3),
    'Change': (tuned_f1 - baseline_f1).round(3)
})

print("F1-Score Comparison:")
print(comparison.to_string(index=False))
print(f"\nBaseline macro F1: {baseline_f1.mean():.3f}")
print(f"Tuned macro F1:    {tuned_f1.mean():.3f}")

F1-Score Comparison:
 Class  Baseline F1  Tuned F1  Change
   DoS        0.854     0.851  -0.003
Normal        0.778     0.770  -0.008
 Probe        0.757     0.715  -0.042
   R2L        0.084     0.010  -0.074
   U2R        0.056     0.130   0.074

Baseline macro F1: 0.506
Tuned macro F1:    0.495


In [9]:
# AGGRESSIVE TUNING — manually weighted, heavily regularized
#
# Instead of 'balanced' (which calculates weights automatically),
# we'll set custom weights that MASSIVELY boost rare classes

print("Training aggressively tuned Random Forest...")
start_time = time.time()

aggressive_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,              # shallower trees = less memorization
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',       # each tree only sees sqrt(41) ≈ 6 features
    class_weight={
        0: 1,     # DoS — already plenty of samples
        1: 1,     # Normal — already plenty
        2: 5,     # Probe — moderate boost
        3: 50,    # R2L — heavy boost
        4: 100    # U2R — massive boost
    },
    random_state=42,
    n_jobs=-1
)

aggressive_rf.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Training complete in {train_time:.1f} seconds")

# Evaluate
y_test_pred_agg = aggressive_rf.predict(X_test)
train_acc = accuracy_score(y_train, aggressive_rf.predict(X_train))
test_acc = accuracy_score(y_test, y_test_pred_agg)

print(f"\nTraining accuracy: {train_acc*100:.2f}%")
print(f"Test accuracy:     {test_acc*100:.2f}%")
print(f"Gap: {(train_acc - test_acc)*100:.2f}%")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT — AGGRESSIVE MODEL (TEST SET)")
print("=" * 60)
print(classification_report(y_test, y_test_pred_agg, target_names=target_encoder.classes_))

Training aggressively tuned Random Forest...
Training complete in 3.1 seconds

Training accuracy: 99.90%
Test accuracy:     74.51%
Gap: 25.40%

CLASSIFICATION REPORT — AGGRESSIVE MODEL (TEST SET)
              precision    recall  f1-score   support

         DoS       0.96      0.77      0.85      7458
      Normal       0.64      0.97      0.77      9711
       Probe       0.84      0.64      0.73      2421
         R2L       0.88      0.02      0.04      2887
         U2R       0.53      0.12      0.20        67

    accuracy                           0.75     22544
   macro avg       0.77      0.51      0.52     22544
weighted avg       0.80      0.75      0.70     22544



In [10]:
# Compare all three models
aggressive_f1 = f1_score(y_test, y_test_pred_agg, average=None)

comparison = pd.DataFrame({
    'Class': target_encoder.classes_,
    'Baseline': baseline_f1.round(3),
    'Tuned': tuned_f1.round(3),
    'Aggressive': aggressive_f1.round(3)
})

print("F1-Score Comparison Across All Models:")
print(comparison.to_string(index=False))
print(f"\nMacro F1 scores:")
print(f"  Baseline:   {baseline_f1.mean():.3f}")
print(f"  Tuned:      {tuned_f1.mean():.3f}")
print(f"  Aggressive: {aggressive_f1.mean():.3f}")

F1-Score Comparison Across All Models:
 Class  Baseline  Tuned  Aggressive
   DoS     0.854  0.851       0.854
Normal     0.778  0.770       0.775
 Probe     0.757  0.715       0.728
   R2L     0.084  0.010       0.045
   U2R     0.056  0.130       0.195

Macro F1 scores:
  Baseline:   0.506
  Tuned:      0.495
  Aggressive: 0.519


In [12]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': aggressive_rf.feature_importances_
}).sort_values('importance', ascending=False)
# Save the best model so far (aggressive — best macro F1)
joblib.dump(aggressive_rf, '../models/random_forest.joblib')
print("Model saved to models/random_forest.joblib")

# Also save the feature importance — useful for the dashboard later
feature_importance.to_csv('../data/feature_importance.csv', index=False)
print("Feature importance saved to data/feature_importance.csv")

Model saved to models/random_forest.joblib
Feature importance saved to data/feature_importance.csv


In [13]:
# What features matter most? This is great for interviews.
print("Top 10 Most Important Features:\n")
for i, row in feature_importance.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['importance']:.4f}")

Top 10 Most Important Features:

  src_bytes                      0.1459
  dst_host_srv_count             0.0768
  logged_in                      0.0709
  service                        0.0668
  dst_bytes                      0.0634
  dst_host_same_src_port_rate    0.0464
  dst_host_serror_rate           0.0462
  dst_host_diff_srv_rate         0.0445
  count                          0.0405
  srv_count                      0.0349
